### Loading, Preprocessing and Chunking PDF File

This notebook loads a pdf file, preprocesses it and chunks so it could be fed into an embedding model.

In [1]:
pip install pinecone

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install langchain

Note: you may need to restart the kernel to use updated packages.


In [22]:
pip install langchain_pinecone

Note: you may need to restart the kernel to use updated packages.


In [4]:
!pip install langchain-ollama

In [5]:
pip install ollama

Note: you may need to restart the kernel to use updated packages.


In [7]:
pip install streamlit

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pinecone-plugin-assistant 1.8.0 requires packaging<25.0,>=24.2, but you have packaging 23.2 which is incompatible.



  Using cached packaging-23.2-py3-none-any.whl.metadata (3.2 kB)
Using cached packaging-23.2-py3-none-any.whl (53 kB)
  Attempting uninstall: packaging
    Found existing installation: packaging 24.2
    Uninstalling packaging-24.2:
      Successfully uninstalled packaging-24.2


In [8]:
pip install langchain-community

Note: you may need to restart the kernel to use updated packages.


In [13]:
from langchain_community.document_loaders import PyPDFLoader

pdf_file = PyPDFLoader("Zambezi_Voice_A_Multilingual_Speech_Corpus_for_Zam.pdf")

data = pdf_file.load()

In [14]:
len(data) 

# output: 5 pages

5

In [15]:
type(data)

# List of Langchain Document objects

list

LangChain uses this structure so that:
- You can chunk documents
- tore them in vector databases
- Keep track of where text came from
- Retrieve original source pages
- Do RAG properly

In [19]:
data[4]

Document(metadata={'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-08-13T20:26:58+02:00', 'author': '', 'title': '', 'subject': '', 'keywords': '', 'moddate': '2023-08-13T20:26:58+02:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'source': 'Zambezi_Voice_A_Multilingual_Speech_Corpus_for_Zam.pdf', 'total_pages': 5, 'page': 4, 'page_label': '3988'}, page_content='9. References\n[1] A. Babu, C. Wang, A. Tjandra, K. Lakhotia, Q. Xu, N. Goyal,\nK. Singh, P. von Platen, Y . Saraf, J. Pino, A. Baevski, A. Conneau,\nand M. Auli, “XLS-R: Self-supervised Cross-lingual Speech Rep-\nresentation Learning at Scale,” in Proc. Interspeech 2022, 2022,\npp. 2278–2282.\n[2] V . Panayotov, G. Chen, D. Povey, and S. Khudanpur, “Lib-\nrispeech: An ASR corpus based on public domain audio books,”\nin ICASSP , IEEE International Conference on Acoustics, Speech\nand Signal Processing - Pr

In [23]:
data[0].page_content

'Zambezi Voice: A Multilingual Speech Corpus for Zambian Languages\nClaytone Sikasote1, Kalinda Siaminwe1, Stanly Mwape1, Bangiwe Zulu1, Mofya Phiri1,\nMartin Phiri1, David Zulu1, Mayumbo Nyirenda1, Antonios Anastasopoulos2\n1University of Zambia, Zambia\n2George Mason University, U.S.A\nclaytone.sikasote@cs.unza.zm\nAbstract\nThis work introduces Zambezi V oice, an open-source multilin-\ngual speech resource for Zambian languages. It contains two\ncollections of datasets: unlabelled audio recordings of radio\nnews and talk shows programs (160 hours) and labelled data\n(over 80 hours) consisting of read speech recorded from text\nsourced from publicly available literature books. The dataset\nis created for speech recognition but can be extended to multi-\nlingual speech processing research for both supervised and un-\nsupervised learning approaches. To our knowledge, this is the\nfirst multilingual speech dataset created for Zambian languages.\nWe exploit pretraining and cross-lingual 

In [25]:
data[0].metadata

{'producer': 'pdfTeX-1.40.21',
 'creator': 'LaTeX with hyperref',
 'creationdate': '2023-08-13T20:26:58+02:00',
 'author': '',
 'title': '',
 'subject': '',
 'keywords': '',
 'moddate': '2023-08-13T20:26:58+02:00',
 'trapped': '/False',
 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2',
 'source': 'Zambezi_Voice_A_Multilingual_Speech_Corpus_for_Zam.pdf',
 'total_pages': 5,
 'page': 0,
 'page_label': '3984'}

In [27]:
len(data[0].page_content)

5701

### 1. Fixed size Chunking
splits text based purely on number of characters

In [47]:
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(
    separator = "\n",
    chunk_size = 1000,
    chunk_overlap = 200
)

In [49]:
chunks = text_splitter.split_documents(data)

In [51]:
print([len(chunk.page_content) for chunk in chunks])

[940, 955, 992, 958, 979, 976, 989, 983, 989, 967, 947, 994, 944, 632, 1000, 960, 940, 939, 997, 977, 949, 979, 951, 960, 959, 973, 980, 950, 556, 984, 968, 951, 991, 974, 989, 980, 761]


In [53]:
len(chunks)

37

In [33]:
chunks[0].page_content

'Zambezi Voice: A Multilingual Speech Corpus for Zambian Languages\nClaytone Sikasote1, Kalinda Siaminwe1, Stanly Mwape1, Bangiwe Zulu1, Mofya Phiri1,\nMartin Phiri1, David Zulu1, Mayumbo Nyirenda1, Antonios Anastasopoulos2\n1University of Zambia, Zambia\n2George Mason University, U.S.A\nclaytone.sikasote@cs.unza.zm\nAbstract\nThis work introduces Zambezi V oice, an open-source multilin-\ngual speech resource for Zambian languages. It contains two\ncollections of datasets: unlabelled audio recordings of radio\nnews and talk shows programs (160 hours) and labelled data\n(over 80 hours) consisting of read speech recorded from text\nsourced from publicly available literature books. The dataset\nis created for speech recognition but can be extended to multi-\nlingual speech processing research for both supervised and un-\nsupervised learning approaches. To our knowledge, this is the\nfirst multilingual speech dataset created for Zambian languages.'

In [35]:
chunks[1].page_content

'lingual speech processing research for both supervised and un-\nsupervised learning approaches. To our knowledge, this is the\nfirst multilingual speech dataset created for Zambian languages.\nWe exploit pretraining and cross-lingual transfer learning by\nfinetuning the Wav2Vec2.0 large-scale multilingual pretrained\nmodel to build end-to-end (E2E) speech recognition models for\nour baseline models. The dataset is released publicly under a\nCreative Commons BY-NC-ND 4.0 license and can be accessed\nthrough the project repository. 1\nIndex Terms: speech recognition, speech translation, low re-\nsource speech processing, Zambian languages\n1. Introduction\nThe development of speech-based systems like automatic\nspeech recognition (ASR) and speech-to-text translation (STT)\nsystems requires data resources (text and speech) on which\nmodels can be trained and tested. In the recent years, the field of\nnatural language processing (NLP) and speech processing (SP)'

### 2. Recursive Chunking
split text using a list of common separator.

In [19]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_text_splitter = RecursiveCharacterTextSplitter(
    separators = ["\n",".", " ", ""],
    chunk_size=500,  
    chunk_overlap=100  
)

In [21]:
chunks = recursive_text_splitter.split_documents(data)
# print(chunks)
print([len(chunk.page_content) for chunk in chunks])

[442, 492, 466, 456, 443, 492, 491, 477, 477, 445, 457, 451, 456, 483, 456, 471, 471, 445, 485, 485, 499, 445, 458, 484, 468, 479, 492, 480, 483, 474, 486, 452, 489, 498, 435, 494, 449, 452, 485, 497, 335, 446, 469, 466, 483, 490, 472, 479, 439, 499, 482, 490, 498, 472, 470, 322, 475, 448, 452, 488, 494, 470, 489, 454, 488, 459, 497, 494, 489, 451, 443, 319]


In [23]:
len(chunks)

72

In [25]:
chunks[0].page_content

'Zambezi Voice: A Multilingual Speech Corpus for Zambian Languages\nClaytone Sikasote1, Kalinda Siaminwe1, Stanly Mwape1, Bangiwe Zulu1, Mofya Phiri1,\nMartin Phiri1, David Zulu1, Mayumbo Nyirenda1, Antonios Anastasopoulos2\n1University of Zambia, Zambia\n2George Mason University, U.S.A\nclaytone.sikasote@cs.unza.zm\nAbstract\nThis work introduces Zambezi V oice, an open-source multilin-\ngual speech resource for Zambian languages. It contains two'

In [27]:
chunks[1].page_content

'gual speech resource for Zambian languages. It contains two\ncollections of datasets: unlabelled audio recordings of radio\nnews and talk shows programs (160 hours) and labelled data\n(over 80 hours) consisting of read speech recorded from text\nsourced from publicly available literature books. The dataset\nis created for speech recognition but can be extended to multi-\nlingual speech processing research for both supervised and un-\nsupervised learning approaches. To our knowledge, this is the'

### Document Embedding 

In [ ]:
pip install python-dotenv

In [ ]:
import os
from pinecone import Pinecone
from dotenv import load_dotenv



# Get your API key at app.pinecone.io


# Instantiate the Pinecone client

load_dotenv()
api_key = os.environ["PINECONE_API_KEY"]  

# Giving our index a name
index_name = "lawpal"


In [41]:
from pinecone import ServerlessSpec, CloudProvider, AwsRegion, Metric

# Create an index if it doesn't exist
if index_name not in pc.list_indexes().names():
   pc.create_index(
        name=index_name,
        dimension=1024,  # Use 1536 for `text-embedding-3-large`
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
index = pc.Index(index_name)
print("Pinecone index initialized!")

Pinecone index initialized!


In [45]:
from langchain_ollama.embeddings import OllamaEmbeddings
from langchain_pinecone import PineconeVectorStore

embeddings  = OllamaEmbeddings(
  model='mxbai-embed-large'
)

vectorstore = PineconeVectorStore(embedding=embeddings, index=index )

vectorstore.add_documents(documents=chunks)

['420a65f1-e8cc-4e5b-aa9a-3c500c1e6566',
 '0dc4950b-4e59-4968-8d9c-48db62a082a0',
 'b02cab27-64ab-44ba-8e3d-d1e15fb6efc0',
 '4bc440fd-77ce-4bc3-b444-49a9fa55c436',
 'eb4c19c9-413c-41c1-b960-caf1821e7631',
 'c3bc4d72-2d6d-459d-b2da-9ffd23786b60',
 '8c353eca-9d4d-444d-ab2c-3290219dc28e',
 '1fb63023-966d-4a67-9650-7566f542f407',
 '1528e929-f5fc-4706-8f8d-256c0b9e972f',
 'dcc13e7a-93f8-4429-b1dc-41b26f49f602',
 'e45e4856-b252-49bc-979d-8664def49552',
 'b09c56ee-7046-471f-991f-1f91c3d1b1d9',
 '95bdd148-d0e2-42fe-a01a-bc82c457b73c',
 '9128d731-e990-4f5e-a06a-1a9ddfa3d761',
 '2aa76130-32b7-4cbf-8faa-a03dd7998d7a',
 '7548dca3-c54c-41d5-89fa-20e0f25cf421',
 '0bb7ee32-7c52-4c17-bfb4-b210b2e056bb',
 '0e7dddb6-8403-4106-99d8-6517f3a06b66',
 '8910901b-eb38-4f0a-80aa-a2aab73311fd',
 'd55ad29e-9e35-4082-975d-20e3f8c07388',
 '897db891-d7f0-42f1-b98c-8392f7e4d0be',
 '764e0979-80bb-4743-8e33-18062ea0d509',
 'af9b9e9c-1c85-485a-b65f-7aff39ec9063',
 '15871588-28f3-4f1b-a9d5-92c483097803',
 '58de2aa0-7010-

Before building a chain, we need three coponents:
1. Retirever for our vector store,
2. a prompt template for combining user question and retrieved context
3. a model to generate the response

search_type: the method of retrieval
search_kwargs: How many chunks to retrieve

retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs = {"k": 2})

In [49]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.1")

retriever = vectorstore.as_retriever(search_type="similarity")

prompt = PromptTemplate.from_template(
    """Use the following context to answer the question.
    If you don't know the answer, say that I don't know.

Context:
{context}

Question:
{question}
"""
)

chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

response = chain.invoke("How many Zambian languages are studied in this research? Mention them")
print(response)

The document mentions that the researchers conducted experiments on five languages: Bemba, Nyanja, Tonga, Lozi, and Lunda. It also states that these languages belong to the Zamibian language cluster.

Later, it is mentioned that Zambia has 73 ethnic/tribal groupings with seven spoken language clusters. However, the researchers specifically focused on five languages within this cluster.

So, to answer your question: There are **5** Zambian languages studied in this research:

1. Bemba
2. Nyanja
3. Tonga
4. Lozi
5. Lunda


We can also see what chunks the retriever is returning to feed into the model

In [51]:
docs = retriever.invoke(
    "How many Zambian languages are studied in this research? Mention them"
)

for i, doc in enumerate(docs):
    print(f"\n--- Document {i+1} ---\n")
    print(doc.page_content)


--- Document 1 ---

eventually also including other minority languages, in this work
the speech data resources we present correspond to only five
languages: Bemba, Nyanja, Tonga, Lozi, and Lunda. We briefly
introduce these languages below:
Bemba (bem) Bemba, also known as IciBemba, is the most
widely spoken language in Zambia with an estimation of over
35% speakers of the entire country population [7]. It is native
to the people of Northern, Luapula, and Muchinga provinces

--- Document 2 ---

eventually also including other minority languages, in this work
the speech data resources we present correspond to only five
languages: Bemba, Nyanja, Tonga, Lozi, and Lunda. We briefly
introduce these languages below:
Bemba (bem) Bemba, also known as IciBemba, is the most
widely spoken language in Zambia with an estimation of over
35% speakers of the entire country population [7]. It is native
to the people of Northern, Luapula, and Muchinga provinces

--- Document 3 ---

Lozi, and Lunda. In a

In [4]:
!pip install pinecone-text --quiet

## Hybrid Vector Search (Dense + Sparse BM25)

The current pipeline uses **dense-only** retrieval (semantic cosine similarity via `mxbai-embed-large`).  
Hybrid search adds a **sparse BM25** component that scores keyword overlap, then linearly combines the two scores:

```
hybrid_score = alpha * dense_score + (1 - alpha) * sparse_score
```

| `alpha` | Behaviour |
|---------|-----------|
| `1.0`   | Pure dense (semantic) – same as before |
| `0.0`   | Pure sparse (BM25 keyword) |
| `0.5`   | Balanced hybrid (recommended starting point) |

### Why it matters for this corpus
The Zambezi Voice paper contains **language-specific technical terms** (e.g. *IciBemba*, *Wav2Vec2.0*, *Lozi*, *E2E ASR*) that a dense model may not have seen in pre-training. BM25 handles exact-term recall, while dense embeddings handle semantic paraphrasing — together they cover both failure modes.

### Step 1 – Requirements

Pinecone's hybrid index requires:
- A **dotproduct** metric index (not cosine)
- `SparseValues` uploaded alongside each dense vector
- `pinecone-text` for the BM25 encoder

In [29]:
import os
from pinecone import Pinecone, ServerlessSpec

# ── Pinecone client (reuse existing key) ────────────────────────────────────
api_key = os.environ.get("PINECONE_API_KEY") or "pcsk_476Mbq_K7Z3aCy8BqmFGYrsbHHwhoeRLcfz9yWg8qbpmZMFBEnCMZR6PsPqPiuzDs9neEe"
pc = Pinecone(api_key=api_key)

# Hybrid indexes MUST use dot-product (not cosine)
hybrid_index_name = "lawpal-hybrid"

if hybrid_index_name not in pc.list_indexes().names():
    pc.create_index(
        name=hybrid_index_name,
        dimension=1024,          # matches mxbai-embed-large output dim
        metric="dotproduct",     # required for hybrid search
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    print(f"Created hybrid index: {hybrid_index_name}")
else:
    print(f"Index '{hybrid_index_name}' already exists — skipping creation.")

hybrid_index = pc.Index(hybrid_index_name)
print("Hybrid index ready.")

Index 'lawpal-hybrid' already exists — skipping creation.
Hybrid index ready.


### Step 2 – Fit the BM25 encoder on the corpus

`BM25Encoder` learns term frequencies from your actual chunks so the sparse scores are tuned to the document vocabulary (including domain-specific Zambian language terms).

In [31]:
from pinecone_text.sparse import BM25Encoder

# Fit BM25 on the text of every chunk produced by the recursive splitter
corpus_texts = [chunk.page_content for chunk in chunks]

bm25 = BM25Encoder()
bm25.fit(corpus_texts)
print(f"BM25 fitted on {len(corpus_texts)} chunks.")


# Optionally persist the encoder so you don't need to refit on reload
# bm25.dump("bm25_params.json")
# bm25 = BM25Encoder().load("bm25_params.json")

  0%|          | 0/72 [00:00<?, ?it/s]

BM25 fitted on 72 chunks.


### Step 3 – Generate both vector types and upsert

For each chunk we produce:
- A **dense vector** (1024 floats) from `mxbai-embed-large`
- A **sparse vector** (`{indices: [...], values: [...]}`) from BM25

Both are sent to Pinecone in a single upsert call.

In [33]:
import uuid
from langchain_ollama.embeddings import OllamaEmbeddings

# Dense embedding model (same as before)
embeddings = OllamaEmbeddings(model="mxbai-embed-large")

def upsert_hybrid_batch(chunks, batch_size=32):
    """Upsert chunks with both dense and sparse vectors in batches."""
    total = len(chunks)
    for start in range(0, total, batch_size):
        batch = chunks[start : start + batch_size]
        texts = [c.page_content for c in batch]

        # Dense embeddings (list of float lists)
        dense_vecs = embeddings.embed_documents(texts)

        # Sparse BM25 vectors
        sparse_vecs = bm25.encode_documents(texts)

        vectors = []
        for i, chunk in enumerate(batch):
            vectors.append({
                "id": str(uuid.uuid4()),
                "values": dense_vecs[i],               # dense
                "sparse_values": sparse_vecs[i],       # sparse (BM25)
                "metadata": {
                    "text": chunk.page_content,
                    **{k: str(v) for k, v in chunk.metadata.items()}
                }
            })

        hybrid_index.upsert(vectors=vectors)
        print(f"Upserted batch {start // batch_size + 1} "
              f"({min(start + batch_size, total)}/{total} chunks)")

upsert_hybrid_batch(chunks)
print("\nAll chunks upserted to hybrid index.")

Upserted batch 1 (32/72 chunks)
Upserted batch 2 (64/72 chunks)
Upserted batch 3 (72/72 chunks)

All chunks upserted to hybrid index.


In [34]:
chunks[0]

Document(metadata={'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-08-13T20:26:58+02:00', 'author': '', 'title': '', 'subject': '', 'keywords': '', 'moddate': '2023-08-13T20:26:58+02:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'source': 'Zambezi_Voice_A_Multilingual_Speech_Corpus_for_Zam.pdf', 'total_pages': 5, 'page': 0, 'page_label': '3984'}, page_content='Zambezi Voice: A Multilingual Speech Corpus for Zambian Languages\nClaytone Sikasote1, Kalinda Siaminwe1, Stanly Mwape1, Bangiwe Zulu1, Mofya Phiri1,\nMartin Phiri1, David Zulu1, Mayumbo Nyirenda1, Antonios Anastasopoulos2\n1University of Zambia, Zambia\n2George Mason University, U.S.A\nclaytone.sikasote@cs.unza.zm\nAbstract\nThis work introduces Zambezi V oice, an open-source multilin-\ngual speech resource for Zambian languages. It contains two')

### Step 4 – Build the hybrid retriever

`alpha` controls the dense-vs-sparse balance.  
Start at `0.5`; tune upward toward `1.0` if paraphrasing queries work well, downward toward `0.0` if exact-term recall matters more.

In [35]:
from langchain_community.retrievers import PineconeHybridSearchRetriever

hybrid_retriever = PineconeHybridSearchRetriever(
    embeddings=embeddings,
    sparse_encoder=bm25,
    index=hybrid_index,
    top_k=4,
    alpha=0.5,
)

print("Hybrid retriever ready.")
print("  alpha =", hybrid_retriever.alpha)
print("  top_k =", hybrid_retriever.top_k)

Hybrid retriever ready.
  alpha = 0.5
  top_k = 4


### Step 5 – Hybrid RAG chain

Drop-in replacement for the previous dense retriever — the rest of the chain is identical.

In [37]:
from langchain_community.retrievers import PineconeHybridSearchRetriever
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda, RunnableParallel
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama

# ── 1. Make sure text_key matches what you stored in metadata ────────────────
# In your upsert cell you stored the chunk text as "text":
#   "metadata": { "text": chunk.page_content, ... }
# So text_key must be "text" (it already is the default, but be explicit):

hybrid_retriever = PineconeHybridSearchRetriever(
    embeddings=embeddings,
    sparse_encoder=bm25,
    index=hybrid_index,
    top_k=4,
    alpha=0.5,
    text_key="text",   # ← must match the metadata key used during upsert
)

# ── 2. Format retrieved Documents into a plain string ────────────────────────
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# ── 3. Build the chain with RunnableParallel (not dict-literal syntax) ───────
llm = ChatOllama(model="llama3.1")

prompt = PromptTemplate.from_template(
    """Use the following context to answer the question.
If you don't know the answer, say that I don't know.

Context:
{context}

Question:
{question}
"""
)

hybrid_chain = (
    RunnableParallel(
        context=hybrid_retriever | RunnableLambda(format_docs),
        question=RunnablePassthrough()
    )
    | prompt
    | llm
    | StrOutputParser()
)

response = hybrid_chain.invoke(
    "How many Zambian languages are studied in this research? Mention them"
)
print(response)

The text does not explicitly state that multiple Zambian languages are being studied. It mentions "Lozi" and "Lunda" as examples of two languages, but it also uses phrases such as "multilingual speech corpus developed for Zambian languages", suggesting a collection of multiple languages.

However, later in the text, it is mentioned that Zambia has 73 ethnic/tribal groupings and seven spoken language clusters. But there is no further information on which specific languages are being studied beyond Lozi and Lunda.

Therefore, based on the available information, I would say:

There are at least two Zambian languages being studied: Lozi and Lunda. The exact number of languages being studied is not specified in the text.


### Step 6 – Inspect retrieved chunks

Confirm the retriever is returning relevant documents and check whether BM25 is surfacing keyword-exact matches.

In [38]:
query = "How many Zambian languages are studied in this research? Mention them"

# Test three alpha values to see the effect of sparse vs dense weighting
for alpha in [0.0, 0.5, 1.0]:
    hybrid_retriever.alpha = alpha
    label = {0.0: "Pure BM25 (sparse)", 0.5: "Hybrid (balanced)", 1.0: "Pure dense"}[alpha]
    docs = hybrid_retriever.invoke(query)
    print(f"\n{'='*60}")
    print(f"  alpha={alpha}  →  {label}")
    print(f"  Retrieved {len(docs)} chunks")
    print(f"{'='*60}")
    for j, doc in enumerate(docs[:2]):   # show first 2 for brevity
        print(f"\n  [Chunk {j+1}]")
        print("  " + doc.page_content[:300].replace("\n", " "))

# Reset to balanced for production use
hybrid_retriever.alpha = 0.5


  alpha=0.0  →  Pure BM25 (sparse)
  Retrieved 4 chunks

  [Chunk 1]
  gual speech dataset for Zambian languages. 3.2. ASR datasets from Radio Broadcasts To mitigate the unavailability of speech recognition datasets, some researchers have used radio broadcasts to build systems. Notable examples are the West African Radio Corpus [10], a multilingual speech dataset conta

  [Chunk 2]
  gual speech resource for Zambian languages. It contains two collections of datasets: unlabelled audio recordings of radio news and talk shows programs (160 hours) and labelled data (over 80 hours) consisting of read speech recorded from text sourced from publicly available literature books. The data

  alpha=0.5  →  Hybrid (balanced)
  Retrieved 4 chunks

  [Chunk 1]
  Lozi, and Lunda. In addition, we conduct baseline experiments by training E2E ASR models using cross-lingual transfer learn- ing to ascertain its potential and usefulness. To our knowledge, this is the first multilingual speech corpus devel

In [57]:
# ── Write the Streamlit app to disk ───────────────────────────────────────────

code = r'''
import os
import streamlit as st
from pinecone import Pinecone
from pinecone_text.sparse import BM25Encoder
from langchain_community.retrievers import PineconeHybridSearchRetriever
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda, RunnableParallel
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama, OllamaEmbeddings

st.set_page_config(page_title="Zambezi Voice RAG", page_icon="🎙️")
st.title("🎙️ Zambezi Voice — Hybrid Search")
st.caption("Pinecone · BM25 · mxbai-embed-large · Llama 3.1")

@st.cache_resource(show_spinner="Loading models and index…")
def build_chain():
    embeddings = OllamaEmbeddings(model="mxbai-embed-large")

    bm25 = BM25Encoder().default()

    api_key = (
        os.environ.get("PINECONE_API_KEY")
        or "pcsk_476Mbq_K7Z3aCy8BqmFGYrsbHHwhoeRLcfz9yWg8qbpmZMFBEnCMZR6PsPqPiuzDs9neEe"
    )
    pc = Pinecone(api_key=api_key)
    hybrid_index = pc.Index("lawpal-hybrid")

    retriever = PineconeHybridSearchRetriever(
        embeddings=embeddings,
        sparse_encoder=bm25,
        index=hybrid_index,
        top_k=4,
        alpha=0.5,
        text_key="text",
    )

    llm = ChatOllama(model="llama3.1")

    prompt = PromptTemplate.from_template(
        """Use the following context to answer the question.
If you don't know the answer, say that you don't know.

Context:
{context}

Question:
{question}
"""
    )

    def format_docs(docs):
        return "\\n\\n".join(doc.page_content for doc in docs)

    chain = (
        RunnableParallel(
            context=retriever | RunnableLambda(format_docs),
            question=RunnablePassthrough(),
        )
        | prompt
        | llm
        | StrOutputParser()
    )
    return chain, retriever


chain, retriever = build_chain()

if "messages" not in st.session_state:
    st.session_state.messages = []

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

if question := st.chat_input("Ask about the Zambezi Voice research…"):
    st.session_state.messages.append({"role": "user", "content": question})

    with st.chat_message("user"):
        st.markdown(question)

    with st.chat_message("assistant"):
        with st.expander("📄 Retrieved chunks", expanded=False):
            docs = retriever.invoke(question)
            for i, doc in enumerate(docs, 1):
                source = doc.metadata.get("source", "unknown")
                st.markdown(f"**Chunk {i}** — `{source}`")
                st.caption(
                    doc.page_content[:400]
                    + ("…" if len(doc.page_content) > 400 else "")
                )

        placeholder = st.empty()
        full_response = ""

        for chunk in chain.stream(question):
            full_response += chunk
            placeholder.markdown(full_response + "▌")

        placeholder.markdown(full_response)

    st.session_state.messages.append({"role": "assistant", "content": full_response})

with st.sidebar:
    st.header("⚙️ Settings")

    if st.button("🗑️ Clear chat", use_container_width=True):
        st.session_state.messages = []
        st.rerun()

    st.divider()
    st.markdown("**Index:** `lawpal-hybrid`")
    st.markdown("**Embeddings:** `mxbai-embed-large`")
    st.markdown("**LLM:** `llama3.1`")
    st.markdown("**top_k:** 4 · **alpha:** 0.5")
    st.markdown("**Metric:** dotproduct")
'''


with open("rag_app.py", "w", encoding="utf-8") as f:
    f.write(code)
print("✅ rag_app.py written")



✅ rag_app.py written


In [55]:
!pip install pyngrok streamlit -q

In [61]:
import subprocess, sys

proc = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "rag_app.py",
     "--server.port", "8501",
     "--server.headless", "true"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
print("✅ Streamlit started — open http://localhost:8501")

✅ Streamlit started — open http://localhost:8501
